In [1]:
import os
import random
import shutil
from PIL import Image
from sklearn.model_selection import train_test_split

# IMPORTANT:
# Notebook is inside notebooks/
# So we use ../ to go to project root

RAW_BASE = "../data/raw/PetImages"
CAT_DIR = os.path.join(RAW_BASE, "Cat")
DOG_DIR = os.path.join(RAW_BASE, "Dog")

PROCESSED_BASE = "../data/processed"

IMG_SIZE = (224, 224)
SEED = 42

random.seed(SEED)

print("CAT_DIR:", CAT_DIR)
print("DOG_DIR:", DOG_DIR)


CAT_DIR: ../data/raw/PetImages\Cat
DOG_DIR: ../data/raw/PetImages\Dog


In [2]:
print("Cat folder exists:", os.path.exists(CAT_DIR))
print("Dog folder exists:", os.path.exists(DOG_DIR))

print("Sample cat files:", os.listdir(CAT_DIR)[:5])
print("Sample dog files:", os.listdir(DOG_DIR)[:5])


Cat folder exists: True
Dog folder exists: True
Sample cat files: ['0.jpg', '1.jpg', '10.jpg', '100.jpg', '1000.jpg']
Sample dog files: ['0.jpg', '1.jpg', '10.jpg', '100.jpg', '1000.jpg']


In [3]:
# Remove old processed folder (if exists)
if os.path.exists(PROCESSED_BASE):
    shutil.rmtree(PROCESSED_BASE)

splits = ["train", "val", "test"]
classes = ["cats", "dogs"]

for split in splits:
    for cls in classes:
        os.makedirs(os.path.join(PROCESSED_BASE, split, cls), exist_ok=True)

print("Created data/processed structure!")


Created data/processed structure!


In [4]:
cat_files = [f for f in os.listdir(CAT_DIR) if f.lower().endswith(".jpg")]
dog_files = [f for f in os.listdir(DOG_DIR) if f.lower().endswith(".jpg")]

print("Total cat images:", len(cat_files))
print("Total dog images:", len(dog_files))


Total cat images: 12499
Total dog images: 12499


In [5]:
def split_files(files, seed=42):
    train, temp = train_test_split(files, test_size=0.2, random_state=seed, shuffle=True)
    val, test = train_test_split(temp, test_size=0.5, random_state=seed, shuffle=True)
    return train, val, test

cat_train, cat_val, cat_test = split_files(cat_files, SEED)
dog_train, dog_val, dog_test = split_files(dog_files, SEED)

print("Cats:", len(cat_train), len(cat_val), len(cat_test))
print("Dogs:", len(dog_train), len(dog_val), len(dog_test))


Cats: 9999 1250 1250
Dogs: 9999 1250 1250


In [6]:
def process_and_save(files, in_dir, out_dir):
    saved = 0
    skipped = 0

    for fname in files:
        in_path = os.path.join(in_dir, fname)
        out_path = os.path.join(out_dir, fname)

        try:
            img = Image.open(in_path).convert("RGB")
            img = img.resize(IMG_SIZE)
            img.save(out_path, format="JPEG", quality=95)
            saved += 1
        except Exception:
            skipped += 1

    return saved, skipped


splits_map = {
    "train": (cat_train, dog_train),
    "val": (cat_val, dog_val),
    "test": (cat_test, dog_test),
}

total_saved = 0
total_skipped = 0

for split, (cats, dogs) in splits_map.items():
    cat_out = os.path.join(PROCESSED_BASE, split, "cats")
    dog_out = os.path.join(PROCESSED_BASE, split, "dogs")

    s1, k1 = process_and_save(cats, CAT_DIR, cat_out)
    s2, k2 = process_and_save(dogs, DOG_DIR, dog_out)

    print(f"{split.upper()} -> cats saved={s1}, skipped={k1} | dogs saved={s2}, skipped={k2}")

    total_saved += (s1 + s2)
    total_skipped += (k1 + k2)

print("\nPreprocessing complete!")
print("Total saved:", total_saved)
print("Total skipped (corrupt images):", total_skipped)


c:\Users\kekolamb\Desktop\MLOPS Assignment\MLOPS-Assignment2\cats-dogs-mlops\.venv\Lib\site-packages\PIL\TiffImagePlugin.py:890: UserWarning: Truncated File Read
  warnings.warn(str(msg))


TRAIN -> cats saved=9999, skipped=0 | dogs saved=9999, skipped=0
VAL -> cats saved=1250, skipped=0 | dogs saved=1250, skipped=0
TEST -> cats saved=1250, skipped=0 | dogs saved=1250, skipped=0

Preprocessing complete!
Total saved: 24998
Total skipped (corrupt images): 0


In [7]:
def count_images(folder):
    return len([f for f in os.listdir(folder) if f.lower().endswith(".jpg")])

for split in ["train", "val", "test"]:
    c = count_images(os.path.join(PROCESSED_BASE, split, "cats"))
    d = count_images(os.path.join(PROCESSED_BASE, split, "dogs"))
    print(split.upper(), "cats:", c, "| dogs:", d)


TRAIN cats: 9999 | dogs: 9999
VAL cats: 1250 | dogs: 1250
TEST cats: 1250 | dogs: 1250
